# 05 · 完整链路：SOG → PyIF 程序 → 编译检查 → MCP 执行 + Jev 判断 → 作答

把前面几本串起来，在 FinMCP 的任务上对比 **ReAct** 和 **Jev-as-IF**：

| 步骤 | Jev-as-IF |
| --- | --- |
| 1 | 按 SOG 披露操作，渲染类型化接口桩 |
| 2 | 大模型**一次**写出 PyIF 程序（语义判断写成 `jev.*`） |
| 3 | 静态检查；不通过则把错误交回大模型修改（最多 2 次） |
| 4 | 解释执行：工具走 MCP 回放，身份参数由运行时注入，`jev.*` 由 Jev 判断，不确定时回退一次大模型 |
| 5 | 运行出错时允许一次修改 |
| 6 | 大模型根据 `answer(...)` 列出的事实作答 |

任务按依赖结构分成"含语义判断"和"只有机械依赖"两组（论文 §6.1 的筛选）。默认各取少量任务，先看链路和量级。

**会调用大模型和 Jev，产生费用**（主要是 ReAct 的多轮调用）。所有请求都有缓存。

注意：回放环境里工具几乎不耗时，所以耗时主要反映大模型和 Jev；Jev-as-IF 调用了标准轨迹以外的参数时，回放可能 `approx` 或 `miss`，结果要结合回放状态一起看。

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json, random, time
import pandas as pd
from jif import config, finmcp, build, sog as S
from jif.llm import LLM, Ledger
from jif.jevrt import JevRuntime, Gate
from jif.mcp_conn import McpConnection
from jif.agents import ReActAgent, JevAsIfAgent, llm_fallback, judge
pd.set_option("display.max_colwidth", 120)

keys = config.load_keys()
print(config.describe(keys))
path = config.ARTIFACTS / "sog_finmcp.json"
sog = S.SOG.load(path) if path.exists() else build.build_sog(finmcp.iter_tasks(("multi_tool", "multi_turn")))[0]

ledger = Ledger()
llm = LLM(keys["LLM_BASE_URL"], keys["LLM_API_KEY"], keys["LLM_MODEL"], ledger=ledger)
conn = McpConnection()
jev = JevRuntime(sog, model=keys["JEV_MODEL"], elaboration="full", gate=Gate(choice_min=0.6, noul_hi=0.7, noul_lo=0.3),
                 ledger=ledger, fallback=llm_fallback(llm))
react = ReActAgent(llm, conn)
jif_agent = JevAsIfAgent(llm, conn, sog, jev, hops=None, static_check=True)

## 选任务

In [ ]:
N_DECISION, N_MECHANICAL, SEED = 5, 3, 7
screened = [(tid, finmcp.screen(t)) for tid, t in finmcp.iter_tasks(("multi_tool",))]
decision = [tid for tid, s in screened if s["decision"]]
mechanical = [tid for tid, s in screened if not s["decision"] and finmcp.tool_calls(finmcp.get_task(tid))]
rng = random.Random(SEED)
rng.shuffle(decision); rng.shuffle(mechanical)
SELECTED = [("含语义判断", t) for t in decision[:N_DECISION]] + [("只有机械依赖", t) for t in mechanical[:N_MECHANICAL]]
print(f"多工具子集：含语义判断 {len(decision)} 条，只有机械依赖 {len(mechanical)} 条")
pd.DataFrame([{"组": g, "任务": t, "问题": finmcp.first_question(finmcp.get_task(t))[:60]} for g, t in SELECTED])

## 运行

In [ ]:
RUN_JUDGE = True
rows, runs = [], {}
for group, tid in SELECTED:
    task = finmcp.get_task(tid)
    q, ident = finmcp.first_question(task), {"userId": finmcp.user_id(task)}
    for agent in (react, jif_agent):
        try:
            r = agent.run(q, tid, ident)
        except Exception as e:
            r = {"method": agent.name, "answer": "", "error": repr(e), "replay": []}
        j = judge(llm, q, finmcp.reference_answer(task), r.get("answer", "")) if RUN_JUDGE else {"score": None, "reason": ""}
        runs[(tid, agent.name)] = r
        status = pd.Series([e["status"] for e in r.get("replay", [])], dtype=object).value_counts().to_dict()
        rows.append({"组": group, "任务": tid, "方法": agent.name, "成功": j["score"], "大模型调用": r.get("llm_calls"),
                     "输入 token": r.get("llm_in"), "输出 token": r.get("llm_out"), "Jev 调用": r.get("jev_calls", 0),
                     "Jev token": r.get("jev_tokens", 0), "工具调用": r.get("tool_calls"), "回放": status,
                     "耗时(s)": r.get("wall"), "错误": (r.get("error") or "")[:80]})
        print(f"{tid:16} {agent.name:9} 成功={j['score']} 大模型调用={r.get('llm_calls')} 工具={r.get('tool_calls')} 错误={(r.get('error') or '-')[:60]}")
df = pd.DataFrame(rows)
df

## 汇总

In [ ]:
num = ["成功", "大模型调用", "输入 token", "输出 token", "Jev 调用", "Jev token", "工具调用", "耗时(s)"]
summary = df.groupby(["组", "方法"])[num].mean().round(2)
display(summary)
ratio = (df[df["方法"] == "Jev-as-IF"].groupby("组")[["大模型调用", "输入 token", "耗时(s)"]].mean()
         / df[df["方法"] == "ReAct"].groupby("组")[["大模型调用", "输入 token", "耗时(s)"]].mean()).round(2)
print("Jev-as-IF / ReAct：")
display(ratio)

## 看一条 Jev-as-IF 运行

In [ ]:
tid = SELECTED[0][1]
r = runs[(tid, "Jev-as-IF")]
print("问题：", finmcp.first_question(finmcp.get_task(tid)))
print("\n程序：\n" + r.get("code", ""))
print("\n静态检查：", r.get("checks"))
print("运行修改次数：", r.get("runtime_repairs"), "| 错误：", r.get("error"))
for d in r.get("decisions", []):
    print(f"  {d['kind']}：{d['question']} → {d['result']}（{d['gated']}）")
print("\n回答：", (r.get("answer") or "")[:800])

## 保存结果

In [ ]:
stamp = time.strftime("%Y%m%d-%H%M%S")
out = config.ARTIFACTS / f"results_{stamp}.jsonl"
with out.open("w", encoding="utf-8") as f:
    for (tid, method), r in runs.items():
        f.write(json.dumps({"task_id": tid, "method": method, **r}, ensure_ascii=False, default=str) + "\n")
df.to_csv(config.ARTIFACTS / f"results_{stamp}.csv", index=False)
print("已保存：", out)
display(pd.DataFrame(Ledger.summarize(ledger.records)).T)
conn.close()